In [1]:
import os
os.chdir(os.path.join(os.getcwd(), '..'))
os.getcwd()

'd:\\learning\\learning\\programing\\portfolio_projects\\1_image'

In [2]:
from src.components.optuna_model_selection.lightning_modules import MRIModule

In [3]:
n = MRIModule([63,35,46])

In [4]:
import torch

In [5]:
n.get_img_masks(torch.ones((3,256,256))).shape

torch.Size([1, 3, 256, 256])

In [12]:
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision.transforms import v2
from src.components.optuna_model_selection.model import MRIFlexAttentionUNet, CorrectionModel
import cv2


from torch.utils.data import Dataset, DataLoader
class BaseModelDataset(Dataset):
    def __init__(self, mri_dir: Path, transform = None):
        
        self.ds = [file for file in mri_dir.iterdir()]
        self.ds.append(self.ds[-1])
        self.ds.insert(0, self.ds[0])
        self.transform = transform

    
    def __len__(self):

        return len(self.ds)-2

            

    def __getitem__(self, index): 
        
        img = cv2.imread(self.ds[index], cv2.IMREAD_UNCHANGED)
        min_img = img.min()
        max_img = img.max()
        img = (img - min_img)/(max_img - min_img + 1e-8)
        img = torch.from_numpy(img).to(torch.float32)

        
        if self.transform:
            img = self.transform(img)
           
        

        return img, index



class MRISegmentationModel:
    def __init__(self, config):


        self.device = config.device

        state_dict = torch.load(config.base_model_path, 
                                'cpu', weights_only=False)['state_dict']
        inner_state_dict = {
            k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")
        }
        
        hparams = torch.load(config.base_model_path, 'cpu', weights_only= False)['hyper_parameters']
        base_model = MRIFlexAttentionUNet(
            inp_channels=hparams['inp_channels'],
            first_conv_out_channels = hparams['first_conv_out_channels'],
            num_classes=3,
            depth = hparams['depth'],
            n_encoder_conv_layers = hparams['n_encoder_conv_layers'],
            n_decoder_conv_layers = hparams['n_decoder_conv_layers'],
            kernel_sizes = hparams['kernel_sizes']
        )
        base_model.load_state_dict(inner_state_dict)


        # state_dict = torch.load(config.correction_model_path, 
        #                         'cpu', weights_only=False)['state_dict']
        # inner_state_dict = {
        #     k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")
        # }
        
        # hparams = torch.load(config.correction_model_path, 'cpu', weights_only= False)['hyper_parameters']
        # correction_model = CorrectionModel(
        #     inp_channels=hparams['inp_channels'],
        #     first_conv_out_channels = hparams['first_conv_out_channels'],
        #     num_classes=3,
        #     depth = hparams['depth'],
        #     n_encoder_conv_layers = hparams['n_encoder_conv_layers'],
        #     n_decoder_conv_layers = hparams['n_decoder_conv_layers'],
        #     kernel_sizes = hparams['kernel_sizes']
        # )

        # correction_model.load_state_dict(inner_state_dict)


        correction_model = CorrectionModel(
                    inp_channels=7,
                    first_conv_out_channels = 32,
                    num_classes=3,
                    depth = 3,
                    n_encoder_conv_layers = 2,
                    n_decoder_conv_layers = 2,
                    kernel_sizes = 3)

        self.base_model = base_model.to(self.device).eval()
        self.correction_model = correction_model.to(self.device).eval()

        if config.base_thresholds_path is not None:
            base_thresholds = np.load(config.base_thresholds_path)
        else:
            base_thresholds = np.array([0.5,0.5,0.5])

        if config.correction_thresholds_path is not None:
            correction_thresholds = np.load(config.correction_thresholds_path)
        else:
            correction_thresholds = np.array([0.5,0.5,0.5])

        self.base_thresholds = torch.as_tensor(base_thresholds, dtype=torch.float32, device=config.device).view(1, -1, 1, 1)
        self.correction_thresholds = torch.as_tensor(correction_thresholds, dtype=torch.float32, device=config.device).view(1, -1, 1, 1)


        self.config = config
        
        
    

    @torch.no_grad()
    def __call__(self, mri_dir):


        base_dataset = BaseModelDataset(mri_dir=mri_dir, transform=self.config.dev_transform)
        inp, _ = next(iter(base_dataset))
        self.out_shape = (3, inp.shape[1], inp.shape[2])
        self.len = len(base_dataset)
        self.base_dataloader = DataLoader(base_dataset, self.config.batch_size, False, num_workers=self.config.num_workers)

        correction_imgs = []
        correction_inp_masks = []
        
        summed_masks = torch.zeros(self.out_shape, dtype = int, device = self.device)

        for inp, idx in self.base_dataloader:
            inp = inp.to(self.device)
            if idx[0] != 0 or idx[-1] != self.len -1:
                correction_imgs.append(inp.cpu())
                torch.permute

            out = self.base_model(inp)
            probs = torch.sigmoid(out) 
            out = out.cpu().to(torch.float16)
            correction_inp_masks.append(out.to(torch.float16))
            correction_inp_masks.append(out)

            summed_masks += (probs >= self.base_thresholds).int().sum(dim = 0)

    

        summed_masks = masks / torch.Tensor([74,61,45], device = self.device).unsqueeze(0).unsqeeze(0)


        final_logits = torch.zeros(self.out_shape, dtype = float, device = self.device)
        for imgs, masks in zip(correction_imgs, correction_inp_masks):
            imgs = imgs.to(self.device)
            masks = masks.to(self.device)
            
            logits = self.correction_model(imgs, masks, summed_masks)
            final_logits = torch.cat([final_logits, logits])

        final_logits =final_logits[1:]
        final_probs = torch.sigmoid(final_logits)
        final_masks = (probs >= self.correction_thresholds).bool().cpu()


        return final_masks


In [7]:
from src.entity.config_entity import MRISegmentationEntity


d:\learning\learning\programing\portfolio_projects\1_image\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
config = MRISegmentationEntity()

Benchmarking optimal batch size...
Batch size 16 is valid


In [13]:
modee = MRISegmentationModel(config)